# PhysioNet EEGBCI Nonlinear Analysis

This is a companion tutorial to the paper *"A Primer on Low-Dimensional Neural Dynamics: PCA-Based Trajectory Analysis for EEG and MEG."*

It extends the linear state-space analysis from the introductory EEGBCI tutorial. We keep the same participants, four left/right hand execution and imagery conditions, preprocessing, and sensor-space representation, but ask a new question:

> **Do nonlinear embeddings reveal local trajectory structure that PCA does not preserve?**

We compare **PCA, UMAP, PHATE, and Isomap** on exactly the same trial-time observations. The goal is not to select the most visually attractive embedding. Instead, we connect every visualization to geometry diagnostics, inspect participant confounding, estimate velocity without crossing trial boundaries, and test whether label-free temporal alignment changes cross-participant consistency.

<div class="alert alert-secondary">
<b>🗺️ Where this notebook fits:</b><br>
<ol style="margin-bottom: 0; margin-top: 5px;">
  <li><b>Introductory EEGBCI tutorial:</b> Builds and validates the core PCA trajectory workflow.</li>
  <li><b>Nonlinear EEGBCI tutorial (this notebook):</b> Compares alternative geometries and alignment assumptions.</li>
  <li><b>EEGBCI decoding tutorial:</b> Tests prediction with subject-disjoint, fold-local preprocessing.</li>
  <li><b>MEG Faces tutorials:</b> Repeat the workflow on whitened 306-sensor MEG, on band-limited envelopes, and on cross-participant decoding.</li>
</ol>
</div>

<div class="alert alert-warning">
<b>⚠️ Why "which embedding looks best" is the wrong question:</b><br>
A nonlinear embedding always returns something, and low-dimensional projections of noise can look strikingly organized. Every figure below is therefore paired with a diagnostic: a geometry score that says whether neighbourhoods survived the projection, a silhouette that says whether participant identity — rather than condition — is what the space is organizing, and a velocity estimate that is forbidden from connecting one trial to the next. A pattern that appears in only one panel and none of the diagnostics is not a result.
</div>

### Analysis roadmap

1. Choose the state-space representation.
2. Preprocess and balance whole trials.
3. Reshape trials into a common sample matrix.
4. Fit four dimensionality-reduction methods.
5. Validate local geometry across neighborhood scales.
6. Diagnose participant versus condition structure.
7. Reconstruct and visualize condition trajectories.
8. Estimate trial-respecting velocity fields.
9. Evaluate temporal Procrustes alignment.
10. Export the complete reproducible analysis bundle.

## 0. Setup & Configuration

As in the introductory notebook, we make all analysis choices visible before touching the data. The notebook executes every transformation directly; it does **not** call the headless script. The companion script mirrors these same operations for unattended runs.

### 0.1. Environment & imports

The analysis uses `coco-pipe` for dimensionality reduction, geometry evaluation, trial-safe velocity, alignment, and visualization. Scikit-learn contributes only the silhouette diagnostic.

<div class="alert alert-info">
<b>💻 Installation:</b><br>
Install this repository with its declared dependencies before running the notebook:<br>
<code>python -m pip install -e .</code>
</div>

In [ ]:
# --- Standard library -------------------------------------------------------
import os
import warnings
from pathlib import Path

# --- Numerical and plotting libraries ---------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score

# --- coco-pipe ---------------------------------------------------------------
from coco_pipe.dim_reduction import DimReduction
from coco_pipe.dim_reduction.evaluation import (
    MethodSelector,
    compute_velocity_fields,
)
from coco_pipe.transforms import TemporalProcrustesAlignment
from coco_pipe.viz.theme import set_coco_theme

# --- Repository data and provenance helpers ---------------------------------
from pca_neural_trajectories import (
    LABEL_NAMES,
    load_eegbci_container,
    setup_data_bids,
    write_manifest,
)

### 0.2. Visual theme, conditions, and analysis parameters

Execution and imagery share colors by effector, while line style distinguishes task mode. This makes the visual grammar identical across every figure.

<div class="alert alert-warning">
<b>⏱️ Computational note:</b><br>
Nonlinear methods are more expensive than PCA. The tutorial therefore selects four trials per participant × condition cell and retains every fourth time sample. Trial selection occurs <i>before</i> temporal decimation, so the analysis unit remains the whole trial. Increase resolution only after completing the default run.
</div>

In [ ]:
# Activate coco-pipe's accessible paper theme globally.
set_coco_theme(mode="paper", colorblind=True)

# Four left/right hand conditions: execution (solid) and imagery (dashed).
CONDITIONS = (3, 4, 5, 6)
CONDITION_COLORS = {3: "#0072B2", 4: "#D55E00", 5: "#0072B2", 6: "#D55E00"}
CONDITION_DASHES = {3: "solid", 4: "solid", 5: "dash", 6: "dash"}

# Analysis choices shared with the headless script.
SUBJECTS = [int(value) for value in os.getenv("EEG_SUBJECTS", "1,2,3,4,5,6,7,8,9,10").split(",")]
ANALYSIS_WINDOW = (-0.2, 1.0)
TRIALS_PER_CELL = int(os.getenv("EEG_NONLINEAR_TRIALS_PER_CELL", "4"))
TIME_STRIDE = int(os.getenv("EEG_NONLINEAR_TIME_STRIDE", "4"))
NEIGHBORHOODS = (10, 20, 40)
METHODS = ("PCA", "UMAP", "PHATE", "Isomap")
FLOW_METHOD = "PHATE"
SEED = 42

BIDS_ROOT = Path(os.getenv("EEG_BIDS_ROOT", "PhysioNet_EEGBCI/BIDS"))
OUTPUT = Path(os.getenv("EEG_NONLINEAR_OUTPUT", "outputs/tutorial_eegbci_nonlinear"))
FIGURES_DIR = OUTPUT / "figures"
REDUCERS_DIR = OUTPUT / "reducers"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REDUCERS_DIR.mkdir(parents=True, exist_ok=True)

# Data preparation is explicit, matching the rest of the tutorial series.
PREPARE_DATA = os.getenv("EEG_PREPARE_DATA", "0") == "1"

print(f"Subjects: {SUBJECTS}")
print(f"BIDS input: {BIDS_ROOT}")
print(f"Output bundle: {OUTPUT}")

## Step 1. Choose the Representation (State Space)

We retain the same state definition used in the introductory analysis: at each time point, one EEG state is the spatial voltage pattern across the 64 scalp electrodes. Each trial is therefore a path through a 64-dimensional sensor space.

For trial `i` and time `t`, the state is a vector with one entry per channel. PCA and the nonlinear reducers all receive this **same sensor vector**. We do not feed PCA scores into UMAP or PHATE, because that would give PCA an unacknowledged filtering role and make the comparison asymmetric.

<div class="alert alert-success">
<b>✅ Fair-comparison rule:</b><br>
Every reducer sees the same trials, time samples, channels, crop, and normalization. Only the geometry-learning algorithm changes.
</div>

### Load, crop, and standardize EEGBCI

The BIDS conversion performs the expensive signal cleaning described in the introductory tutorial. Here we load broad epochs, crop to the predeclared −0.2–1.0 s task window, and z-score each channel across observations and time.

The broad load window protects the earlier preprocessing from epoch-edge effects; it is **not** included in reducer fitting.

In [ ]:
# Prepare/download only when explicitly requested. Existing BIDS data are reused.
if PREPARE_DATA:
    setup_data_bids(
        subjects=SUBJECTS,
        runs=list(range(3, 15)),
        root=BIDS_ROOT,
    )

sensor = load_eegbci_container(
    BIDS_ROOT,
    subjects=[f"{subject:03d}" for subject in SUBJECTS],
    conditions=CONDITIONS,
    runs=[f"{run:02d}" for run in range(3, 15)],
    tmin=ANALYSIS_WINDOW[0] - 1.0,
    tmax=ANALYSIS_WINDOW[1] + 1.5,
    baseline=(-1.0, 0.0),
)

full_times = np.asarray(sensor.coords["time"], dtype=float)
analysis_mask = (full_times >= ANALYSIS_WINDOW[0] - 1e-8) & (
    full_times <= ANALYSIS_WINDOW[1] + 1e-8
)
sensor = sensor.isel(time=analysis_mask).zscore(dim=("obs", "time"), eps=1e-5)

In [ ]:
# Confirm the biological structure before flattening anything.
preprocessing_summary = pd.Series({
    "trials loaded": sensor.X.shape[0],
    "channels": sensor.X.shape[1],
    "time samples in analysis window": sensor.X.shape[2],
    "first time (s)": float(sensor.coords["time"][0]),
    "last time (s)": float(sensor.coords["time"][-1]),
})
preprocessing_summary.to_frame("value")

## Step 2. Balance Whole Trials Across Participants and Conditions

Nonlinear embeddings are density-sensitive: cells that contribute more observations can exert more influence on the learned geometry. Because every trial contributes many time samples, even a small trial imbalance is multiplied after reshaping.

We first retain participants who have all four conditions. Then we find the smallest participant × condition cell and select the same number of **whole trials** from every cell. The random generator is seeded, so rerunning the analysis selects the same trials.

<div class="alert alert-warning">
<b>⚠️ Avoid pseudobalancing:</b><br>
Do not flatten first and randomly sample time points. That can fragment trials, distort temporal coverage, and make the velocity analysis impossible to interpret.
</div>

In [ ]:
all_subjects = np.asarray(sensor.coords["subject"]).astype(str)
all_labels = np.asarray(sensor.y).astype(int)

complete_subjects = [
    subject
    for subject in np.unique(all_subjects)
    if all(
        np.any((all_subjects == subject) & (all_labels == condition))
        for condition in CONDITIONS
    )
]
if not complete_subjects:
    raise RuntimeError("No subject has trials for every requested condition.")

cells = {
    (subject, condition): np.flatnonzero(
        (all_subjects == subject) & (all_labels == condition)
    )
    for subject in complete_subjects
    for condition in CONDITIONS
}
available_per_cell = min(len(rows) for rows in cells.values())
selected_per_cell = min(TRIALS_PER_CELL, available_per_cell)

rng = np.random.default_rng(SEED)
selected_rows = []
balance_records = []
for (subject, condition), rows in sorted(cells.items()):
    chosen = np.sort(rng.choice(rows, selected_per_cell, replace=False))
    selected_rows.append(chosen)
    balance_records.append({
        "subject": subject,
        "condition": condition,
        "condition_name": LABEL_NAMES[condition],
        "available_trials": len(rows),
        "selected_trials": len(chosen),
    })

balanced_idx = np.concatenate(selected_rows)
time_idx = np.arange(0, sensor.X.shape[2], TIME_STRIDE)
sensor = sensor.isel(obs=balanced_idx.tolist(), time=time_idx.tolist())
balance_table = pd.DataFrame(balance_records)

In [ ]:
display(balance_table)

balance_check = balance_table.pivot(
    index="subject", columns="condition_name", values="selected_trials"
)
print(f"Selected {selected_per_cell} trials per complete cell.")
balance_check

## Step 3. Reshape into a Common Samples × Features Matrix

Each reducer expects a two-dimensional matrix. We transpose the balanced array from `(trial, channel, time)` to `(trial, time, channel)`, then join trial and time into one sample axis:

```text
(trial, channel, time) → (trial, time, channel) → (trial × time, channel)
```

Alongside the matrix, we construct parallel vectors for participant, condition, trial identity, and within-trial time. These vectors are what allow us to reconstruct trajectories later and prevent false temporal connections during velocity estimation.

In [ ]:
X = np.asarray(sensor.X, dtype=float)
times = np.asarray(sensor.coords["time"], dtype=float)
trial_subjects = np.asarray(sensor.coords["subject"]).astype(str)
trial_labels = np.asarray(sensor.y).astype(int)
trial_ids = np.asarray(sensor.ids).astype(str)

samples = X.transpose(0, 2, 1).reshape(-1, X.shape[1])
sample_subjects = np.repeat(trial_subjects, len(times))
sample_labels = np.repeat(trial_labels, len(times))
trial_sequence = np.repeat(np.arange(len(X)), len(times))
within_trial_time = np.tile(times, len(X))

assert len(samples) == len(sample_subjects) == len(sample_labels)
assert len(samples) == len(trial_sequence) == len(within_trial_time)

In [ ]:
sample_summary = pd.Series({
    "balanced trials": len(X),
    "time samples per trial": len(times),
    "rows given to every reducer": len(samples),
    "sensor features per row": samples.shape[1],
    "unique trial groups": len(np.unique(trial_sequence)),
})
sample_summary.to_frame("value")

## Step 4. Fit PCA, UMAP, PHATE, and Isomap

The four methods encode different geometric assumptions:

| Method | Geometry emphasized | Main interpretive strength | Main caution |
|---|---|---|---|
| PCA | Global linear variance | Stable axes, loadings, and variance accounting | Cannot unfold curved structure |
| UMAP | Local fuzzy neighborhoods | Compact local organization | Global distances and density can be distorted |
| PHATE | Diffusion geometry | Smooth continua and branches | Scale and axes remain arbitrary |
| Isomap | Geodesic distances on a neighbor graph | Approximate manifold-wide paths | Sensitive to graph shortcuts/disconnections |

All methods produce three coordinates for visualization. We fix random seeds where the algorithms are stochastic. The `reducers` dictionary contains fitted `coco-pipe.DimReduction` objects that will later be scored and saved.

<div class="alert alert-danger">
<b>🚫 No common coordinate ruler:</b><br>
A distance of 1 in PHATE space is not equivalent to a distance of 1 in PCA or UMAP space. Cross-method comparisons must use dimensionless validation metrics, not raw embedded distances.
</div>

In [ ]:
valid_k = [k for k in NEIGHBORHOODS if k < len(samples)]
if not valid_k:
    raise RuntimeError("The balanced sample is too small for neighborhood evaluation.")

neighbor_count = min(20, len(samples) - 1)
reducers = {
    "PCA": DimReduction("PCA", n_components=3, random_state=SEED),
    "UMAP": DimReduction(
        "UMAP", n_components=3, n_neighbors=neighbor_count, random_state=SEED
    ),
    "PHATE": DimReduction("PHATE", n_components=3, random_state=SEED),
    "Isomap": DimReduction(
        "Isomap", n_components=3, n_neighbors=neighbor_count
    ),
}

flat_embeddings = {}
trajectories = {}
for name, reducer in reducers.items():
    print(f"Fitting {name} on {samples.shape} ...")
    embedding = np.asarray(reducer.fit_transform(samples))
    reducer.score(
        embedding,
        X=samples,
        metrics=["trustworthiness", "continuity"],
        k_values=valid_k,
        max_eval_samples=min(3000, len(samples)),
    )
    flat_embeddings[name] = embedding
    trajectories[name] = embedding.reshape(len(X), len(times), 3)

pd.DataFrame({name: values.shape for name, values in trajectories.items()}, index=["trials", "times", "dimensions"])

## Step 5. Validate Local Geometry Across Neighborhood Scales

A nonlinear embedding can look smooth while inventing neighbors or tearing apart relationships present in sensor space. We therefore evaluate two complementary rank-based metrics:

1. <span style="color: #0072B2;"><b>Trustworthiness:</b></span> penalizes **false neighbors**—points that become close only after embedding.
2. <span style="color: #D55E00;"><b>Continuity:</b></span> penalizes **missing neighbors**—points that were close in sensor space but become separated.

Both range from 0 to 1, with larger values indicating better preservation. We evaluate `k = 10, 20, 40` because an embedding can succeed at one neighborhood scale and fail at another.

<div class="alert alert-warning">
<b>🧠 Interpretation boundary:</b><br>
High trustworthiness or continuity supports geometric fidelity. It does not establish a biological manifold, an attractor, condition discriminability, or generalization to unseen participants.
</div>

In [ ]:
quality_records = MethodSelector(reducers).collect().to_frame()
quality_summary = (
    quality_records.groupby(["method", "metric"], as_index=False)["value"]
    .mean()
    .sort_values(["metric", "value"], ascending=[True, False])
)

display(quality_records)
display(quality_summary)

In [ ]:
quality_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Trustworthiness", "Continuity"),
    shared_yaxes=True,
)
for column, metric in enumerate(("trustworthiness", "continuity"), start=1):
    for method in METHODS:
        rows = quality_records[
            (quality_records["method"] == method)
            & (quality_records["metric"] == metric)
        ].sort_values("scope_value")
        quality_figure.add_trace(
            go.Scatter(
                x=rows["scope_value"],
                y=rows["value"],
                mode="lines+markers",
                name=method,
                legendgroup=method,
                showlegend=column == 1,
            ),
            row=1,
            col=column,
        )
quality_figure.update_yaxes(range=[0, 1], title_text="score", row=1, col=1)
quality_figure.update_xaxes(title_text="neighborhood size k")
quality_figure.update_layout(
    title="Local geometry across neighborhood scales", height=480, width=980
)
quality_figure.show()

## Step 6. Diagnose Participant Versus Condition Structure

EEG sensor patterns differ across participants because of anatomy, electrode placement, impedance, and individual physiology. A well-separated embedding may therefore organize **who produced the data** rather than **which motor condition was performed**.

We compute silhouette scores twice in each fitted embedding: once using participant labels and once using condition labels. Labels are applied only after unsupervised fitting.

- Values near **1** indicate compact, separated groups.
- Values near **0** indicate overlapping groups.
- Negative values indicate that many samples lie closer to another group.

<div class="alert alert-danger">
<b>⚠️ Descriptive, not inferential:</b><br>
Time samples from the same trial are dependent. These silhouettes diagnose what structures an embedding; they are not p-values, cross-validated decoding scores, or independent-observation effect estimates.
</div>

In [ ]:
silhouette_sample_size = min(2000, len(samples) - 1)
diagnostic_records = []
for name, embedding in flat_embeddings.items():
    diagnostic_records.append({
        "method": name,
        "subject_silhouette": silhouette_score(
            embedding,
            sample_subjects,
            sample_size=silhouette_sample_size,
            random_state=SEED,
        ),
        "condition_silhouette": silhouette_score(
            embedding,
            sample_labels,
            sample_size=silhouette_sample_size,
            random_state=SEED,
        ),
    })

embedding_diagnostics = pd.DataFrame(diagnostic_records)
embedding_diagnostics

In [ ]:
diagnostic_figure = go.Figure()
diagnostic_figure.add_bar(
    x=embedding_diagnostics["method"],
    y=embedding_diagnostics["subject_silhouette"],
    name="participant",
)
diagnostic_figure.add_bar(
    x=embedding_diagnostics["method"],
    y=embedding_diagnostics["condition_silhouette"],
    name="condition",
)
diagnostic_figure.update_layout(
    barmode="group",
    title="What structures each embedding: participant or condition?",
    yaxis_title="silhouette score",
)
diagnostic_figure.show()

## Step 7. Reconstruct and Plot Condition-Mean Trajectories

Reducer fitting ignored temporal order: each row was simply one sensor state. We now reverse the reshape and recover `(trial, time, dimension)`. Averaging trials within condition produces four centroid trajectories in each fitted space.

Execution is shown with solid lines and imagery with dashed lines. Blue denotes left hand; vermilion denotes right hand.

<div class="alert alert-success">
<b>👀 What can be compared:</b><br>
Within each panel, inspect branching, loops, return paths, and relative condition separation. Across panels, compare qualitative topology only. Axis direction, absolute distance, and apparent speed are not on a common scale.
</div>

In [ ]:
trajectory_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=METHODS,
    horizontal_spacing=0.10,
    vertical_spacing=0.14,
)
for panel, method in enumerate(METHODS):
    row, column = divmod(panel, 2)
    for condition in CONDITIONS:
        mean_trajectory = trajectories[method][trial_labels == condition].mean(axis=0)
        trajectory_figure.add_trace(
            go.Scatter(
                x=mean_trajectory[:, 0],
                y=mean_trajectory[:, 1],
                mode="lines",
                line={
                    "color": CONDITION_COLORS[condition],
                    "dash": CONDITION_DASHES[condition],
                    "width": 3,
                },
                name=LABEL_NAMES[condition],
                legendgroup=str(condition),
                showlegend=panel == 0,
            ),
            row=row + 1,
            col=column + 1,
        )
trajectory_figure.update_xaxes(showticklabels=False, title_text="dimension 1")
trajectory_figure.update_yaxes(showticklabels=False, title_text="dimension 2")
trajectory_figure.update_layout(
    title="Condition-mean trajectories: compare shape, not coordinate scale",
    height=850,
    width=1000,
)
trajectory_figure.show()

## Step 8. Estimate Trial-Respecting Velocity Fields

A trajectory's instantaneous displacement is approximated from consecutive time samples within a trial. If we simply difference the flattened matrix, the last state of trial 1 would be connected to the first state of trial 2—a fictitious transition that never occurred.

`compute_velocity_fields` receives two safeguards:

1. `groups=trial_sequence` identifies which samples belong to the same trial.
2. `times=within_trial_time` provides the correct temporal coordinate within each trial.

The within-trial differences are then locally pooled in PHATE space. We summarize vector norms by condition and draw a sparse arrow field to avoid overplotting.

<div class="alert alert-warning">
<b>🧭 Descriptive flow, not a fitted dynamical system:</b><br>
The arrows summarize observed local displacements. They do not estimate a differential equation, prove an attractor, or support raw speed comparisons with another embedding. Trial endpoints naturally have no forward difference.
</div>

In [ ]:
flow_embedding = flat_embeddings[FLOW_METHOD]
velocity = compute_velocity_fields(
    X=samples,
    X_emb=flow_embedding,
    groups=trial_sequence,
    times=within_trial_time,
    n_neighbors=neighbor_count,
)
velocity_norm = np.linalg.norm(velocity, axis=1)

velocity_summary = pd.DataFrame([
    {
        "condition": condition,
        "condition_name": LABEL_NAMES[condition],
        "median_velocity_norm": float(
            np.median(velocity_norm[sample_labels == condition])
        ),
        "p95_velocity_norm": float(
            np.percentile(velocity_norm[sample_labels == condition], 95)
        ),
        "zero_velocity_fraction": float(
            np.mean(velocity_norm[sample_labels == condition] == 0)
        ),
    }
    for condition in CONDITIONS
])
velocity_summary

In [ ]:
# Scale arrows for legibility within PHATE; this scale has no physical units.
nonzero = velocity_norm > 0
plot_scale = 1.0
if nonzero.any():
    span = np.ptp(flow_embedding[:, :2], axis=0)
    plot_scale = 0.04 * float(np.max(span)) / float(
        np.percentile(velocity_norm[nonzero], 90)
    )

velocity_figure = go.Figure()
shown = np.linspace(
    0, len(flow_embedding) - 1, min(180, len(flow_embedding)), dtype=int
)
for index in shown:
    if velocity_norm[index] == 0:
        continue
    condition = int(sample_labels[index])
    velocity_figure.add_trace(go.Scatter(
        x=[
            flow_embedding[index, 0],
            flow_embedding[index, 0] + velocity[index, 0] * plot_scale,
        ],
        y=[
            flow_embedding[index, 1],
            flow_embedding[index, 1] + velocity[index, 1] * plot_scale,
        ],
        mode="lines",
        line={"color": CONDITION_COLORS[condition], "width": 1},
        opacity=0.35,
        showlegend=False,
    ))
velocity_figure.update_layout(
    title=f"Trial-safe descriptive flow in {FLOW_METHOD} space",
    xaxis_title="dimension 1",
    yaxis_title="dimension 2",
)
velocity_figure.show()

## Step 9. Evaluate Temporal Procrustes Alignment

The silhouette diagnostic may reveal that participant identity dominates pooled geometry. Temporal Procrustes alignment tests a specific explanation: participants may share a temporal pattern that is expressed in differently rotated participant-specific PCA spaces.

The transform performs three operations:

1. Fit a PCA separately to each participant.
2. Build a shared temporal reference from pooled PCA.
3. Learn a label-free orthogonal rotation/reflection from each participant space to that reference.

To isolate the contribution of alignment, the control is **participant PCA before rotation**, not raw sensor space. Comparing sensor space with aligned PCA would confound dimensionality reduction and alignment.

<div class="alert alert-danger">
<b>🔒 Transductive-analysis warning:</b><br>
This descriptive fit uses all displayed participants. It must not be interpreted as performance on unseen participants. Predictive analyses require fold-local reference fitting and label-free test-participant calibration, as implemented in the decoding tutorial.
</div>

In [ ]:
alignment_components = min(10, X.shape[1])
alignment = TemporalProcrustesAlignment(
    n_components=alignment_components, random_state=SEED
)
aligned = alignment.fit_transform(X, groups=trial_subjects)

# Reconstruct each participant's PCA scores before applying its rotation.
unaligned = np.empty_like(aligned)
for subject in np.unique(trial_subjects):
    rows = trial_subjects == subject
    participant = X[rows]
    pooled = participant.transpose(0, 2, 1).reshape(-1, X.shape[1])
    scores = alignment.subject_pcas_[subject].transform(pooled)
    unaligned[rows] = scores.reshape(
        len(participant), len(times), alignment_components
    ).transpose(0, 2, 1)

unaligned_trajectories = np.moveaxis(unaligned, 1, 2)
aligned_trajectories = np.moveaxis(aligned, 1, 2)

In [ ]:
# Correlate each participant's mean temporal pattern with all others.
consistency_rows = []
for representation, scores in (
    ("participant PCA", unaligned_trajectories),
    ("participant PCA + Procrustes", aligned_trajectories),
):
    for subject in np.unique(trial_subjects):
        own = scores[trial_subjects == subject].mean(axis=0).ravel()
        other = scores[trial_subjects != subject].mean(axis=0).ravel()
        consistency_rows.append({
            "representation": representation,
            "subject": subject,
            "correlation": float(np.corrcoef(own, other)[0, 1]),
        })

alignment_consistency = pd.DataFrame(consistency_rows)
alignment_summary = (
    alignment_consistency.groupby("representation")["correlation"]
    .agg(["mean", "sem"])
    .reset_index()
)
display(alignment_consistency)
display(alignment_summary)

In [ ]:
alignment_figure = go.Figure(go.Bar(
    x=alignment_summary["representation"],
    y=alignment_summary["mean"],
    error_y={"type": "data", "array": alignment_summary["sem"]},
))
alignment_figure.update_layout(
    title="Alignment sensitivity: leave-one-participant trajectory consistency",
    yaxis_title="correlation",
)
alignment_figure.show()

aligned_trajectory_figure = go.Figure()
for condition in CONDITIONS:
    mean_trajectory = aligned_trajectories[trial_labels == condition].mean(axis=0)
    aligned_trajectory_figure.add_trace(go.Scatter(
        x=mean_trajectory[:, 0],
        y=mean_trajectory[:, 1],
        mode="lines",
        line={
            "color": CONDITION_COLORS[condition],
            "dash": CONDITION_DASHES[condition],
            "width": 3,
        },
        name=LABEL_NAMES[condition],
    ))
aligned_trajectory_figure.update_layout(
    title="Condition trajectories after label-free temporal alignment",
    xaxis_title="aligned PC1",
    yaxis_title="aligned PC2",
)
aligned_trajectory_figure.show()

## Step 10. Export the Complete Reproducible Analysis

A reproducible analysis saves more than its headline figure. We preserve:

- the participant × condition balance table;
- every neighborhood-level quality score and summary;
- participant/condition silhouette diagnostics;
- velocity and alignment summaries;
- all fitted reducer objects;
- sensor, embedding, velocity, and alignment arrays;
- interactive HTML figures and a standalone combined report;
- a manifest containing the full configuration, package versions, and Git commits.

PNG and SVG copies are attempted through Kaleido. If its browser backend is unavailable, the notebook records that fact in the manifest and still completes the HTML, table, model, and array exports.

In [ ]:
tables = {
    "balance": balance_table,
    "quality_records": quality_records,
    "quality_summary": quality_summary,
    "embedding_diagnostics": embedding_diagnostics,
    "velocity_summary": velocity_summary,
    "alignment_consistency": alignment_consistency,
    "alignment_summary": alignment_summary,
}
figures = {
    "quality_by_neighborhood": quality_figure,
    "embedding_diagnostics": diagnostic_figure,
    "method_trajectories": trajectory_figure,
    "velocity_field": velocity_figure,
    "alignment_consistency": alignment_figure,
    "aligned_trajectories": aligned_trajectory_figure,
}

for name, table in tables.items():
    table.to_csv(OUTPUT / f"{name}.csv", index=False)

static_export_error = None
for name, figure in figures.items():
    figure.write_html(FIGURES_DIR / f"{name}.html", include_plotlyjs="cdn")
    if static_export_error is None:
        try:
            figure.write_image(FIGURES_DIR / f"{name}.png", scale=2)
            figure.write_image(FIGURES_DIR / f"{name}.svg")
        except Exception as error:
            static_export_error = f"{type(error).__name__}: {error}"
            warnings.warn(
                "Static Plotly export is unavailable; continuing with HTML figures. "
                "Check the Kaleido browser installation.",
                stacklevel=2,
            )

for name, reducer in reducers.items():
    reducer.save(REDUCERS_DIR / f"{name.lower()}.pkl")

In [ ]:
np.savez_compressed(
    OUTPUT / "analysis_arrays.npz",
    sensor=X,
    times=times,
    subjects=trial_subjects,
    labels=trial_labels,
    trial_ids=trial_ids,
    unaligned=unaligned_trajectories,
    aligned=aligned_trajectories,
    velocity=velocity,
    **{f"embedding_{name.lower()}": values for name, values in trajectories.items()},
)

manifest = {
    "subjects_requested": SUBJECTS,
    "subjects_analyzed": complete_subjects,
    "bids_root": str(BIDS_ROOT),
    "conditions": list(CONDITIONS),
    "analysis_window": list(ANALYSIS_WINDOW),
    "trials_per_cell_requested": TRIALS_PER_CELL,
    "trials_per_cell_used": selected_per_cell,
    "time_stride": TIME_STRIDE,
    "neighborhoods": valid_k,
    "flow_method": FLOW_METHOD,
    "alignment_components": alignment_components,
    "random_state": SEED,
    "shape": list(X.shape),
    "static_figure_exports_complete": static_export_error is None,
    "static_figure_export_error": static_export_error,
}
write_manifest(OUTPUT / "analysis_manifest.json", manifest, status="complete")

html_parts = [
    "<h1>EEGBCI nonlinear neural trajectories</h1>",
    "<p>All methods use the same balanced trials and time samples. "
    "Nonlinear coordinate scales are not compared directly.</p>",
]
for index, (name, figure) in enumerate(figures.items()):
    html_parts.append(f"<h2>{name.replace('_', ' ').title()}</h2>")
    html_parts.append(figure.to_html(
        full_html=False, include_plotlyjs="inline" if index == 0 else False
    ))
for name, table in tables.items():
    html_parts.append(f"<h2>{name.replace('_', ' ').title()}</h2>")
    html_parts.append(table.to_html(index=False))
(OUTPUT / "report.html").write_text("\n".join(html_parts), encoding="utf-8")

print(f"Saved nonlinear analysis → {OUTPUT}")

In [ ]:
saved_files = sorted(
    str(path.relative_to(OUTPUT))
    for path in OUTPUT.rglob("*")
    if path.is_file()
)
pd.DataFrame({"saved_file": saved_files})

## Conclusions & Interpretation Checklist

A nonlinear trajectory figure becomes scientifically useful only when its assumptions and diagnostics travel with it. Before interpreting a pattern, ask:

- **Was the design balanced before flattening?** Every participant × condition cell should contribute the same number of whole trials.
- **Is the relevant geometry preserved?** Inspect trustworthiness and continuity across more than one neighborhood size.
- **What organizes the space?** A dominant participant silhouette weakens a pooled condition narrative.
- **Am I comparing only within-space quantities?** Nonlinear axes, distances, and velocity scales are arbitrary across methods.
- **Did temporal calculations respect trial boundaries?** Trial endpoints must never connect to the next trial.
- **What assumption does alignment add?** Improved consistency supports a shared rotated temporal representation, not identical neural generators.

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
Use nonlinear embeddings as complementary views of sensor-space geometry. Prefer conclusions that remain coherent across neighborhood validation, participant diagnostics, temporal reconstruction, and alignment sensitivity—not conclusions that depend on one visually striking panel.
</div>

## Running the Same Analysis Headlessly

The notebook above is the teaching implementation: every intermediate operation is visible and inspectable. For a non-interactive run, the companion script repeats the same sequence and saves the same output contract:

```bash
python scripts/analysis_eegbci_nonlinear.py --skip-prepare
```

Omit `--skip-prepare` when the script should explicitly prepare the EEGBCI BIDS conversion first.